### **🧪 What are Custom Generic Tests (in dbt)?**

**Custom generic tests are reusable, parameterized data tests** that **you write once and apply to many models/columns** using **YAML**, just like built-in dbt tests (`not_null`, `unique`, `relationships`).

They extend dbt’s testing framework so you can **enforce business-specific rules** that don’t exist out of the box.

----------------

**🧠 Simple Definition**

> A custom generic test is a **macro-based test** that accepts arguments and can be applied declaratively in `schema.yml` across multiple datasets.

-------------------

**🔍 Where This Fits in dbt**

- **Built-in generic tests** → provided by dbt

- **Custom generic tests** → written by you (macros)

- **Singular tests** → one-off SQL queries

Custom generic tests sit **between** built-in tests and singular tests:
**reusable + configurable + scalable**.

--------------------

#### **🆚 Generic vs Custom Generic vs Singular**

| Type               | Reusable | Defined In | Written As              |
| ------------------ | -------- | ---------- | ----------------------- |
| Built-in Generic   | ✅        | dbt        | YAML                    |
| **Custom Generic** | ✅        | You        | **Macro (SQL + Jinja)** |
| Singular           | ❌        | You        | SQL file                |


-----------------


**🏗️ How Custom Generic Tests Work (Internals)**

A **generic test** macro must:

- Be placed in `macros/`

- Start with `test_`

- Return **rows that violate the rule**

👉 If **0 rows returned → test passes**

👉 If **rows returned → test fails**

---------------


#### **📁 Folder Structure (Best Practice)**

In [ ]:
dbt_project/
├── macros/
│   └── tests/
│       └── test_positive_values.sql
├── models/
│   └── schema.yml

----------

**🧩 Example 1: Column Must Be Positive**

**1️⃣ Custom Generic Test Macro**

In [ ]:
-- macros/tests/test_positive_values.sql
{% test positive_values(model, column_name) %}

select *
from {{ model }}
where {{ column_name }} <= 0

{% endtest %}

📌 This returns rows where the column is **not positive.**

-------------------

**2️⃣ Use It in `schema.yml`**

In [ ]:
models:
  - name: fct_payments
    columns:
      - name: amount
        tests:
          - positive_values

🎯 Applied declaratively, just like `not_null`.

------------------

**🧩 Example 2: Date Should Not Be in the Future**

**Macro**

In [ ]:
{% test not_in_future(model, column_name) %}

select *
from {{ model }}
where {{ column_name }} > current_date

{% endtest %}

**YAML Usage**

In [ ]:
columns:
  - name: order_date
    tests:
      - not_in_future

**🧩 Example 3: Parameterized Test (Advanced & Real-World)**

**Macro with Arguments**

In [ ]:
{% test max_value(model, column_name, max_allowed) %}

select *
from {{ model }}
where {{ column_name }} > {{ max_allowed }}

{% endtest %}

**YAML Usage**

In [ ]:
columns:
  - name: discount_percentage
    tests:
      - max_value:
          max_allowed: 100

--------

**🧠 Why Not Just Use Singular Tests?**

| Reason      | Explanation                        |
| ----------- | ---------------------------------- |
| Reusability | Apply across many models           |
| Clean YAML  | Business rules live next to schema |
| Consistency | Same logic everywhere              |
| Scalability | Easy to maintain                   |


--------------

#### **🔄 Custom Generic Tests & Data Contracts**

Custom generic tests are **key building blocks** of **data contracts**:

- Contract rule: “amount must be positive”

- Enforcement: custom generic test

- Failure → contract violation 🚨

This is widely used in teams working with dbt Labs best practices.

---------------

#### **🚀 When Should You Write Custom Generic Tests?**

Write one when:

- Rule is **business-specific**

- Rule applies to **multiple models**

- Built-in tests are insufficient


❌ Don’t write them for:

- One-time checks → use singular tests

- Simple null/unique checks → built-ins exist

----------

**🧠 Best Practices**

✅ Prefix macro with test_

✅ Return **only failing rows**

✅ Keep logic simple

✅ Store in `macros/tests/`

✅ Name tests clearly (business meaning)

✅ Reuse everywhere